# 02 — AI-Assisted Sentiment Labelling and Manual Check

This notebook:

1. Loads the manually inspected sentence-level dataset.
2. Uses `facebook/bart-large-mnli` for zero-shot labelling.
3. Flags low-confidence predictions for manual review.
4. Exports the AI-labelled file.
5. Loads the manually checked labels.
6. Converts the final dataset to binary Positive/Negative labels.

The AI label is treated as **annotation assistance**, not as ground truth. The final label should be manually checked.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..")
INTERMEDIATE_DIR = PROJECT_ROOT / "data" / "intermediate"
LABELED_DIR = PROJECT_ROOT / "data" / "labeled"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
LABELED_DIR.mkdir(parents=True, exist_ok=True)

manual_path = INTERMEDIATE_DIR / "02_sentence_level_manual_inspection.csv"
fallback_path = INTERMEDIATE_DIR / "01_sentence_level.csv"

if manual_path.exists():
    sentence_df = pd.read_csv(manual_path)
    print("Loaded manually inspected file:", manual_path)
else:
    sentence_df = pd.read_csv(fallback_path)
    print("WARNING: Manual inspection file not found.")
    print("Using:", fallback_path)

display(sentence_df.head())
print("Rows:", len(sentence_df))

## Install and load the zero-shot model

In [ ]:
%pip install -q transformers torch tqdm

In [ ]:
from tqdm.auto import tqdm
from transformers import pipeline

tqdm.pandas()

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

candidate_labels = [
    "Positive",
    "Negative",
    "Neutral"
]

## Define AI labelling function

In [ ]:
def ai_label_text(text):
    text = str(text).strip()

    if text == "":
        return pd.Series(["Neutral", 0.0])

    result = classifier(
        text,
        candidate_labels,
        hypothesis_template="This employee review expresses {} sentiment."
    )

    return pd.Series([
        result["labels"][0],
        result["scores"][0]
    ])

In [ ]:
text_df = sentence_df.copy()

text_df[["AI_Label", "AI_Confidence"]] = (
    text_df["sentence_text"]
    .progress_apply(ai_label_text)
)

text_df["Review_Status"] = text_df["AI_Confidence"].apply(
    lambda score: "Manual Review" if score < 0.75 else "Accepted"
)

text_df["Final_Label"] = text_df["AI_Label"]

display(
    text_df[
        ["sentence_text", "AI_Label", "AI_Confidence",
         "Review_Status", "Final_Label"]
    ].head(20)
)

## Export AI-labelled data

Manually inspect the labels, especially rows with `Review_Status = Manual Review`.

You may also revise high-confidence rows when the prediction is semantically wrong.

After checking, save the edited file as:

`data/labeled/04_ai_labeled_reviews_manual_checked.csv`

In [ ]:
ai_output = LABELED_DIR / "03_ai_labeled_reviews.csv"
manual_checked_output = LABELED_DIR / "04_ai_labeled_reviews_manual_checked.csv"

text_df.to_csv(ai_output, index=False, encoding="utf-8-sig")

print("AI-labelled file saved:", ai_output)
print("After manual checking, save as:")
print(manual_checked_output)

## Load manually checked labels

In [ ]:
if not manual_checked_output.exists():
    raise FileNotFoundError(
        f"Manual checked file not found: {manual_checked_output}\n"
        "Open 03_ai_labeled_reviews.csv, review Final_Label, then save the checked file."
    )

checked_df = pd.read_csv(manual_checked_output)

print("Loaded:", manual_checked_output)
display(checked_df.head())
print(checked_df["Final_Label"].value_counts(dropna=False))

## Standardise final labels

In [ ]:
label_map = {
    1: "Positive",
    2: "Negative",
    3: "Neutral",
    "1": "Positive",
    "2": "Negative",
    "3": "Neutral"
}

checked_df["Final_Label"] = checked_df["Final_Label"].apply(
    lambda x: label_map.get(x, x)
)

print(checked_df["Final_Label"].value_counts(dropna=False))

## Build final labelled text dataset

The final modelling task in the project is binary sentiment classification, so Neutral rows are removed after manual checking.

In [ ]:
labelled_df = checked_df.copy()

drop_cols = [
    c for c in ["AI_Label", "AI_Confidence", "Review_Status"]
    if c in labelled_df.columns
]
labelled_df = labelled_df.drop(columns=drop_cols)

labelled_df = labelled_df.rename(columns={
    "Final_Label": "sentiment",
    "sentence_text": "review_text",
    "Review Text": "review_text"
})

required_cols = ["review_text", "sentiment"]
labelled_df = labelled_df.dropna(subset=required_cols).reset_index(drop=True)

print("Before removing Neutral:", len(labelled_df))

labelled_df = labelled_df[
    labelled_df["sentiment"].isin(["Positive", "Negative"])
].reset_index(drop=True)

print("After removing Neutral:", len(labelled_df))
print(labelled_df["sentiment"].value_counts())

display(labelled_df.head())

## Save raw labelled sentence dataset

In [ ]:
output_path = INTERMEDIATE_DIR / "05_sentence_raw.csv"
labelled_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved:", output_path)